In [1]:
import numpy as np
import math
from scipy.integrate import odeint
import sympy as sym
from sympy import diff
from sympy.matrices import Matrix
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
import matplotlib.pyplot as plt

In [5]:
#0.5*np.pi
#print(np.deg2rad(90))

1.5707963267948966


In [14]:
init_vprinting()

In [15]:
# joint configuration and velocity are functions of time
t = sym.symbols("t")
#q = sym.Array([sym.Function("x1")(t), sym.Function("θ2")(t), sym.Function("θ3")(t)])
#dq = q.diff(t)

In [16]:
q1, q2, q3 = sym.symbols('q1 q2 q3')
dq1, dq2, dq3 = sym.symbols('dq1 dq2 dq3')
q = [q1, q2, q3]
dq = [dq1, dq2, dq3]

In [17]:
# link masses
M1 = 3
M2 = 2
M3 = 1

# link lengths
LX = 0.1
LY = 0.1
L2 = 1
L3 = 0.5

# moments of inertia (base link doesn't rotate so don't need it's inertia)
I2 = M2 * L2 ** 2 / 12
I3 = M3 * L3 ** 2 / 12

# inertia matrices
G1 = np.diag(np.array([M1, M1, 0]))
G2 = np.diag(np.array([M2, M2, I2]))
G3 = np.diag(np.array([M3, M3, I3]))

# gravity
G = 9.8

In [18]:
P1 = sym.Matrix([q[0], 0, 0])
P2 = P1 + sym.Matrix([
        LX + 0.5 * L2 * sym.cos(q[1]),
        LY + 0.5 * L2 * sym.sin(q[1]),
        q[1],
    ])
P3 = P2 + sym.Matrix([
        0.5 * L2 * sym.cos(q[1]) + 0.5 * L3 * sym.cos(q[1] + q[2]),
        0.5 * L2 * sym.sin(q[1]) + 0.5 * L3 * sym.sin(q[1] + q[2]),
        q[2],
    ])

In [34]:
    # link Jacobians
J1 = P1.jacobian(q)
J2 = P2.jacobian(q)
J3 = P3.jacobian(q)

In [91]:
M = J1.transpose() * G1 * J1 + J2.transpose() * G2 * J2 + J3.transpose() * G3 * J3
M

⎡              6.0                                                             ↪
⎢                                                                              ↪
⎢                                                                            2 ↪
⎢-2.0⋅sin(q₂) - 0.25⋅sin(q₂ + q₃)          (-1.0⋅sin(q₂) - 0.25⋅sin(q₂ + q₃))  ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎣       -0.25⋅sin(q₂ + q₃)         -0.25⋅(-1.0⋅sin(q₂) - 0.25⋅sin(q₂ + q₃))⋅si ↪

↪      -2.0⋅sin(q₂) - 0.25⋅sin(q₂ + q₃)                                        ↪
↪                                                                              ↪
↪                                     2          2              2              ↪
↪  + (1.0⋅cos(q₂) + 0.25⋅cos(q₂ + q₃))  + 0.5⋅sin (q₂) + 0.5⋅cos (q₂) + 0.1875 ↪
↪                                                                              ↪
↪                          

In [81]:
    # gravity vector
V = G * (M1 * P1[1] + M2 * P2[1] + M3 * P3[1])
g = V.diff(sym.Matrix(q))




In [25]:
mass_matrix = sym.lambdify([q], M)

In [29]:
n = len(q)
n

In [51]:
Gamma = [[[sym.Integer(0) for _ in range(n)] for _ in range(n)] for _ in range(n)]
for i in range(n):
    for j in range(n):
        for k in range(n):
            term1 = sym.diff(M[i, j], q[k])
            term2 = sym.diff(M[i, k], q[j])
            term3 = sym.diff(M[j, k], q[i])
            Gamma[i][j][k] = 0.5 * (term1 + term2 + term3)
C = sym.zeros(n, n)
for i in range(n):
    for j in range(n):
        c_ij = 0
        for k in range(n):
            c_ij += Gamma[i][j][k] * dq[k]
        C[i, j] = sym.simplify(c_ij)
C

⎡                               0                                  -2.0⋅dq₂⋅co ↪
⎢                                                                              ↪
⎢-2.0⋅dq₂⋅cos(q₂) - 0.25⋅dq₂⋅cos(q₂ + q₃) - 0.25⋅dq₃⋅cos(q₂ + q₃)    -2.0⋅dq₁⋅ ↪
⎢                                                                              ↪
⎣                 -0.25⋅(dq₂ + dq₃)⋅cos(q₂ + q₃)                     -0.25⋅dq₁ ↪

↪ s(q₂) - 0.25⋅dq₂⋅cos(q₂ + q₃) - 0.25⋅dq₃⋅cos(q₂ + q₃)                 -0.25⋅ ↪
↪                                                                              ↪
↪ cos(q₂) - 0.25⋅dq₁⋅cos(q₂ + q₃) - 0.25⋅dq₃⋅sin(q₃)     -0.25⋅dq₁⋅cos(q₂ + q₃ ↪
↪                                                                              ↪
↪ ⋅cos(q₂ + q₃) - 0.25⋅dq₂⋅sin(q₃) - 0.25⋅dq₃⋅sin(q₃)             -0.25⋅dq₁⋅co ↪

↪ (dq₂ + dq₃)⋅cos(q₂ + q₃)               ⎤
↪                                        ⎥
↪ ) - 0.25⋅dq₂⋅sin(q₃) - 0.25⋅dq₃⋅sin(q₃)⎥
↪                                        ⎥
↪ s(q₂ + q₃) - 0

In [86]:
tau = sym.Matrix([0, 0, 0]) 
rhs = tau - C*sym.Matrix(dq) - g

Metodo de inversion

In [87]:
ddq_expr = M.inv() * rhs

In [125]:
mass_matrix = sym.lambdify((q), M, 'numpy')
coriolis_matrix = sym.lambdify(((q), (dq)), C, 'numpy')
gravity_vector = sym.lambdify((q), g, 'numpy')

In [ ]:
mass_matrix(2,0.1,0.3)

In [ ]:
gravity_vector(2,0.1,0.3)

In [ ]:
coriolis_matrix((2,0.1,0.3),(0.2,0,0))

In [131]:
state_vars = [q1, q2, q3, dq1, dq2, dq3]

In [132]:
#rhs_vector =sym.lambdify(((q), (dq)),rhs,'numpy')

rhs_vector =sym.lambdify(state_vars,rhs,'numpy')

In [ ]:
rhs_vector(1, 1, 1, 0.1, 0.1, 0.1)

In [ ]:
q_ddot = np.linalg.solve(mass_matrix, rhs_vector)